In [7]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/Train.csv")
print(df.shape)           # (68654, 16)
print(df.target.mean())   # 0.0183 → 1.83% defaulted


(68654, 16)
0.0183237684621435


In [8]:
num = ["Total_Amount", "duration", "Lender_portion_Funded"]
cat = ["New_versus_Repeat", "loan_type", "lender_id", "country_id"]

X = df[num + cat].copy()
X["lender_id"] = X["lender_id"].astype(str)   # it's an ID, not a quantity
y = df["target"]
groups = df["customer_id"]                    # used to split, not as a feature


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer

pre = ColumnTransformer([
    ("num", Pipeline([
        ("log", FunctionTransformer(np.log1p)),  # tame huge amounts
        ("scale", StandardScaler()),             # mean 0, std 1
    ]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),  # categories → 0/1 columns
])


In [10]:
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("pre", pre),
    ("lr", LogisticRegression(class_weight="balanced", max_iter=1000)),
])


In [11]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score

results = []
for fold, (tr, va) in enumerate(GroupKFold(n_splits=5).split(X, y, groups)):
    model.fit(X.iloc[tr], y.iloc[tr])                # learn on 4/5
    p = model.predict_proba(X.iloc[va])[:, 1]        # predict the hidden 1/5
    results.append({
        "fold": fold,
        "roc_auc": roc_auc_score(y.iloc[va], p),
        "pr_auc": average_precision_score(y.iloc[va], p),
    })

results = pd.DataFrame(results)
print(results.round(3))
print(results.mean().round(3))


   fold  roc_auc  pr_auc
0     0    0.794   0.167
1     1    0.812   0.206
2     2    0.785   0.181
3     3    0.796   0.229
4     4    0.774   0.190
fold       2.000
roc_auc    0.792
pr_auc     0.194
dtype: float64
